# SLC46A1 engraftment, step 3: direct coupling analysis on human PCFT

Builds a deep alignment for Q96NT5 and computes direct couplings between every pair of
positions by mean-field direct coupling analysis with average product correction. The
statistic is derived from evolutionary covariation and not from a language model, so it
is the non-learned readout the line requires beside the ESM-2 likelihood.

The couplings are then read at the pairs the earlier steps identified: the coupled set of
primary substitutions, the measured compensatory pairs on human PCFT, and the measured
cross-linking pairs of the extracellular gate.

Every cell aborts on a failed precondition. Results are written to Drive.

In [ ]:
import os, sys, json, time, hashlib, datetime

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'dca_step3')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/pharmacophore_position_set.tsv', 'structures/7BC6_to_human.json',
            'data/zhao2016_gate.json']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))

RUNLOG = os.path.join(OUT, 'run.log')

def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))

log('start, project=%s' % PROJECT)


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)
fa = os.path.join(SEQ, 'Q96NT5.fasta')
if not os.path.exists(fa) or os.path.getsize(fa) == 0:
    urllib.request.urlretrieve('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', fa)

human = ''.join(l.strip() for l in open(fa) if not l.startswith('>'))
if len(human) != 459:
    raise SystemExit('FATAL Q96NT5 length %d, expected 459' % len(human))
log('query length %d' % len(human))


In [ ]:
import tarfile, re

A3M = os.path.join(OUT, 'Q96NT5.a3m')
HOST = 'https://api.colabfold.com'

def post(path, data):
    req = urllib.request.Request(HOST + path, data=data.encode(),
                                 headers={'User-Agent': 'pcft-dca'})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())

def get(path):
    req = urllib.request.Request(HOST + path, headers={'User-Agent': 'pcft-dca'})
    with urllib.request.urlopen(req, timeout=300) as r:
        return r.read()

if not os.path.exists(A3M) or os.path.getsize(A3M) == 0:
    body = 'q=%s&mode=env' % ('>Q96NT5' + chr(10) + human)
    sub = post('/ticket/msa', body)
    tid = sub.get('id')
    if not tid:
        raise SystemExit('FATAL MSA server did not return a ticket: %s' % sub)
    log('msa ticket %s' % tid)
    for _ in range(360):
        st = json.loads(get('/ticket/' + tid))
        if st.get('status') == 'COMPLETE':
            break
        if st.get('status') in ('ERROR', 'UNKNOWN'):
            raise SystemExit('FATAL MSA job failed: %s' % st)
        time.sleep(10)
    else:
        raise SystemExit('FATAL MSA job did not complete within an hour')
    raw = os.path.join(OUT, 'msa.tar.gz')
    with open(raw, 'wb') as fh:
        fh.write(get('/result/download/' + tid))
    with tarfile.open(raw) as tf:
        names = [n for n in tf.getnames() if n.endswith('.a3m')]
        if not names:
            raise SystemExit('FATAL no a3m in the MSA archive: %s' % tf.getnames())
        with open(A3M, 'wb') as out:
            out.write(tf.extractfile(names[0]).read())

if os.path.getsize(A3M) < 10000:
    raise SystemExit('FATAL alignment is only %d bytes' % os.path.getsize(A3M))
log('a3m %d bytes' % os.path.getsize(A3M))


In [ ]:
import numpy as np

AA = '-ACDEFGHIKLMNPQRSTVWY'
IDX = {a: i for i, a in enumerate(AA)}
Q = len(AA)
L = len(human)
MAX_SEQS = 40000

seqs = [l.rstrip() for l in open(A3M) if l.strip() and not l.startswith('>')]
aln = []
for s in seqs:
    kept = ''.join(c for c in s if not c.islower())
    if len(kept) == L:
        aln.append([IDX.get(c.upper(), 0) for c in kept])
A = np.array(aln, dtype=np.int8)
if A.shape[0] < 500:
    raise SystemExit('FATAL only %d usable sequences in the alignment' % A.shape[0])
if A.shape[0] > MAX_SEQS:
    sel = np.linspace(0, A.shape[0] - 1, MAX_SEQS).astype(int)
    sel[0] = 0
    A = A[sel]
B = A.shape[0]
log('alignment matrix %d sequences x %d columns' % A.shape)

oh = np.zeros((B, L, Q), dtype=np.float32)
oh[np.arange(B)[:, None], np.arange(L)[None, :], A] = 1.0
flat = oh.reshape(B, L * Q)

THETA = 0.2
cut = (1.0 - THETA) * L
neigh = np.zeros(B, dtype=np.int64)
CH = 512
for i in range(0, B, CH):
    sim = flat[i:i + CH] @ flat.T
    neigh[i:i + CH] = (sim >= cut).sum(axis=1)
w = 1.0 / np.maximum(neigh, 1).astype(np.float64)
Meff = float(w.sum())
log('effective sequences %.1f from %d' % (Meff, B))


In [ ]:
PSEUDO = 0.5
wsum = w.sum()
w32 = w.astype(np.float32)

fi = (oh * w32[:, None, None]).sum(axis=0) / wsum
fi = (1 - PSEUDO) * fi + PSEUDO / Q

cnt = (flat * w32[:, None]).T @ flat
cnt /= wsum
pair = cnt.reshape(L, Q, L, Q)[:, 1:, :, 1:]
pair = (1 - PSEUDO) * pair + PSEUDO / (Q * Q)
pair = pair.astype(np.float64)
for i in range(L):
    pair[i, :, i, :] = np.diag(fi[i, 1:])

C = (pair - fi[:, 1:, None, None] * fi[None, None, :, 1:]).reshape(L * (Q - 1), L * (Q - 1))
del pair, cnt
J = -np.linalg.inv(C + 1e-6 * np.eye(C.shape[0], dtype=np.float64))
del C
J = J.reshape(L, Q - 1, L, Q - 1)

DI = np.sqrt((J ** 2).sum(axis=(1, 3)))
np.fill_diagonal(DI, 0.0)
mean_i = DI.mean(axis=1)
S = DI - np.outer(mean_i, mean_i) / DI.mean()
np.fill_diagonal(S, 0.0)
del J
log('coupling matrix computed over %d pairs' % (L * (L - 1) // 2))
np.save(os.path.join(OUT, 'coupling_matrix.npy'), S)


In [ ]:
ck2hu = {int(k): v for k, v in
         json.load(open(os.path.join(PROJECT, 'structures', '7BC6_to_human.json')))['ck2hu'].items()}
pos = [l.rstrip(chr(10)).split(chr(9))
       for l in open(os.path.join(PROJECT, 'data', 'pharmacophore_position_set.tsv'))]
pc = {k: i for i, k in enumerate(pos[0])}
primaries = [int(r[pc['human_pos']]) for r in pos[1:] if r[pc['differs']] == '1']

iu = np.triu_indices(L, 1)
allv = S[iu]
order = np.argsort(allv)
rank_of = np.empty_like(allv)
rank_of[order] = np.arange(len(allv))
pct = {}
for k, (i, j) in enumerate(zip(*iu)):
    pct[(int(i) + 1, int(j) + 1)] = 100.0 * rank_of[k] / (len(allv) - 1)

def percentile(a, b):
    key = (min(a, b), max(a, b))
    return pct.get(key)

out = {'n_sequences': int(B), 'n_usable': int(A.shape[0]), 'meff': Meff,
       'length': L, 'theta': THETA, 'pseudocount': PSEUDO,
       'method': 'mean-field DCA with average product correction',
       'primary_pairs': [], 'compensatory_pairs': [], 'gate_pairs': []}

for x in range(len(primaries)):
    for y in range(x + 1, len(primaries)):
        a, b = primaries[x], primaries[y]
        out['primary_pairs'].append({'i': a, 'j': b, 'sep': abs(a - b),
                                     'score': float(S[a - 1, b - 1]),
                                     'percentile': percentile(a, b)})

for a, b, label in [(392, 196, 'F392V with S196L'), (392, 305, 'F392D with G305L'),
                    (109, 305, 'D109A with G305L')]:
    out['compensatory_pairs'].append({'i': a, 'j': b, 'label': label,
                                      'sep': abs(a - b),
                                      'score': float(S[a - 1, b - 1]),
                                      'percentile': percentile(a, b)})

gate = json.load(open(os.path.join(PROJECT, 'data', 'zhao2016_gate.json')))
measured = set(tuple(sorted(p.replace('C', '').split('/')))
               for p in gate['spontaneous_disulfide'] + gate['induced_by_cuph_and_cadmium'])
gpos = [g['pos'] for g in gate['gate_positions']]
for x in range(len(gpos)):
    for y in range(x + 1, len(gpos)):
        a, b = gpos[x], gpos[y]
        key = tuple(sorted([str(a), str(b)]))
        out['gate_pairs'].append({'i': a, 'j': b, 'sep': abs(a - b),
                                  'score': float(S[a - 1, b - 1]),
                                  'percentile': percentile(a, b),
                                  'measured': any(set(k) == set(key) for k in
                                                  [tuple(sorted([''.join(filter(str.isdigit, u))
                                                                 for u in m])) for m in measured])})

with open(os.path.join(OUT, 'coupling_report.json'), 'w') as fh:
    json.dump(out, fh, indent=1)

n_hi = sum(1 for p in out['primary_pairs'] if p['percentile'] and p['percentile'] >= 99)
log('wrote coupling_report.json; primary pairs %d of which %d above the 99th percentile'
    % (len(out['primary_pairs']), n_hi))
log('compensatory pair percentiles: %s'
    % ', '.join('%s %.1f' % (c['label'], c['percentile']) for c in out['compensatory_pairs']))
